# Calculate Metrics

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import pickle
import json
import os
from pathlib import Path
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Setup paths
DATA_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/data/all_recipes_final.csv"
MODELS_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/Saved_models"
GROUND_TRUTH_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/eval_ground_truth.jsonl"
OUTPUT_DIR = r"E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl"

# Similarity threshold for relevant items
SIMILARITY_THRESHOLD = 0.1  # Items with similarity >= 0.1 are considered relevant
# Set to 0.0 to include all items with positive similarity

# Create output directory if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Models path: {MODELS_PATH}")
print(f"Ground truth path: {GROUND_TRUTH_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Similarity threshold: {SIMILARITY_THRESHOLD}")

## 1. Load Ground Truth, Data (tương tự file 4) và Predictions

In [ ]:
# Ground truth (json)
ground_truth = []
with open(GROUND_TRUTH_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        ground_truth.append(json.loads(line.strip()))

In [ ]:
# query_ids (list)
query_ids = [item['query_id'] for item in ground_truth]
print(f"Loaded {len(query_ids)} query_ids from ground truth")
print(f"First 5 query_ids: {query_ids[:5]}")

In [ ]:
# Load data (all recipes - 10k candidate pool)
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} recipes")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# Add recipe_id column (using index as recipe_id)
df['recipe_id'] = df.index
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# Load ground truth (already loaded earlier, but reload for clarity)
ground_truth_dict = {}
for item in ground_truth:
    query_id = item['query_id']
    # Create mapping: doc_id -> rel score
    relevant_docs = {}
    for doc in item['top10']:
        relevant_docs[doc['doc_id']] = doc['rel']
    ground_truth_dict[query_id] = relevant_docs

print(f"Loaded ground truth for {len(ground_truth_dict)} queries")
print(f"\nExample ground truth for query {query_ids[0]}:")
print(f"Relevant docs: {ground_truth_dict[query_ids[0]]}")

## Step 2: Implement Metrics Functions

In [ ]:
def precision_at_k(predictions, ground_truth_dict, k=10):
    """
    Precision@K: Tỉ lệ items trong top-K có nằm trong ground truth
    """
    precisions = []
    
    for pred in predictions:
        query_id = pred['query_id']
        pred_docs = [item['doc_id'] for item in pred['top10'][:k]]
        
        if query_id not in ground_truth_dict:
            continue
            
        relevant_docs = set(ground_truth_dict[query_id].keys())
        
        # Count hits in top-K
        hits = sum(1 for doc_id in pred_docs if doc_id in relevant_docs)
        precision = hits / k if k > 0 else 0
        precisions.append(precision)
    
    return np.mean(precisions) if precisions else 0.0

In [ ]:
def recall_at_k(predictions, ground_truth_dict, k=10):
    """
    Recall@K: Tỉ lệ relevant items được tìm thấy trong top-K
    """
    recalls = []
    
    for pred in predictions:
        query_id = pred['query_id']
        pred_docs = [item['doc_id'] for item in pred['top10'][:k]]
        
        if query_id not in ground_truth_dict:
            continue
            
        relevant_docs = set(ground_truth_dict[query_id].keys())
        total_relevant = len(relevant_docs)  # Chỉ có 10 items
        
        # Count hits
        hits = sum(1 for doc_id in pred_docs if doc_id in relevant_docs)
        recall = hits / total_relevant if total_relevant > 0 else 0
        recalls.append(recall)
    
    return np.mean(recalls) if recalls else 0.0

In [ ]:
def mrr(predictions, ground_truth_dict):
    """
    Mean Reciprocal Rank: 1/rank của relevant item đầu tiên
    """
    reciprocal_ranks = []
    
    for pred in predictions:
        query_id = pred['query_id']
        pred_docs = [item['doc_id'] for item in pred['top10']]
        
        if query_id not in ground_truth_dict:
            continue
            
        relevant_docs = set(ground_truth_dict[query_id].keys())
        
        # Find first relevant item
        for rank, doc_id in enumerate(pred_docs, 1):
            if doc_id in relevant_docs:
                reciprocal_ranks.append(1.0 / rank)
                break
        else:
            # No relevant item found
            reciprocal_ranks.append(0.0)
    
    return np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0

In [ ]:
def ndcg_at_k(predictions, ground_truth_dict, k=10):
    """
    Normalized Discounted Cumulative Gain @K
    
    DCG = sum(rel_i / log2(i+1)) for i in 1..k
    IDCG = DCG của ideal ranking (sort by rel desc)
    nDCG = DCG / IDCG
    """
    ndcg_scores = []
    
    for pred in predictions:
        query_id = pred['query_id']
        pred_docs = [item['doc_id'] for item in pred['top10'][:k]]
        
        if query_id not in ground_truth_dict:
            continue
            
        relevant_docs = ground_truth_dict[query_id]  # doc_id -> rel
        
        # Calculate DCG
        dcg = 0.0
        for rank, doc_id in enumerate(pred_docs, 1):
            rel = relevant_docs.get(doc_id, 0)  # 0 if not in GT
            dcg += rel / np.log2(rank + 1)
        
        # Calculate IDCG (ideal ranking)
        ideal_rels = sorted(relevant_docs.values(), reverse=True)[:k]
        idcg = 0.0
        for rank, rel in enumerate(ideal_rels, 1):
            idcg += rel / np.log2(rank + 1)
        
        # nDCG
        if idcg > 0:
            ndcg_scores.append(dcg / idcg)
        else:
            ndcg_scores.append(0.0)
    
    return np.mean(ndcg_scores) if ndcg_scores else 0.0

In [ ]:
def map_at_k(predictions, ground_truth_dict, k=10):
    """
    Mean Average Precision @K
    
    AP = (1/|rel|) * sum(P@i * rel(i)) for i where rel(i)=1
    """
    ap_scores = []
    
    for pred in predictions:
        query_id = pred['query_id']
        pred_docs = [item['doc_id'] for item in pred['top10'][:k]]
        
        if query_id not in ground_truth_dict:
            continue
            
        relevant_docs = set(ground_truth_dict[query_id].keys())
        total_relevant = len(relevant_docs)
        
        if total_relevant == 0:
            continue
        
        # Calculate Average Precision
        num_hits = 0
        sum_precisions = 0.0
        
        for rank, doc_id in enumerate(pred_docs, 1):
            if doc_id in relevant_docs:
                num_hits += 1
                precision_at_rank = num_hits / rank
                sum_precisions += precision_at_rank
        
        ap = sum_precisions / total_relevant if total_relevant > 0 else 0.0
        ap_scores.append(ap)
    
    return np.mean(ap_scores) if ap_scores else 0.0

## Step 3: Calculate Metrics for All Methods

In [ ]:
# Calculate metrics for all methods
results = {}

for method_name in ["TFIDF", "Ingredient_TFIDF", "Keyword", "Hybrid"]:
    print(f"\n{'='*60}")
    print(f"Evaluating {method_name}...")
    print('='*60)
    
    preds = all_predictions[method_name]
    
    # Calculate all metrics
    metrics = {
        'Precision@5': precision_at_k(preds, ground_truth_dict, k=5),
        'Precision@10': precision_at_k(preds, ground_truth_dict, k=10),
        'Recall@5': recall_at_k(preds, ground_truth_dict, k=5),
        'Recall@10': recall_at_k(preds, ground_truth_dict, k=10),
        'MRR': mrr(preds, ground_truth_dict),
        'nDCG@5': ndcg_at_k(preds, ground_truth_dict, k=5),
        'nDCG@10': ndcg_at_k(preds, ground_truth_dict, k=10),
        'MAP@5': map_at_k(preds, ground_truth_dict, k=5),
        'MAP@10': map_at_k(preds, ground_truth_dict, k=10),
    }
    
    results[method_name] = metrics
    
    # Print metrics
    for metric_name, value in metrics.items():
        print(f"{metric_name:15s}: {value:.4f}")

print(f"\n{'='*60}")
print("All metrics calculated!")
print('='*60)

## Step 4: Comparison Table

In [ ]:
# Create comparison DataFrame
import pandas as pd

results_df = pd.DataFrame(results).T
results_df = results_df.round(4)

print("\n" + "="*80)
print("COMPARISON TABLE - ALL METHODS")
print("="*80)
print(results_df.to_string())
print("="*80)

# Find best method for each metric
print("\n🏆 Best Method per Metric:")
print("-" * 60)
for metric in results_df.columns:
    best_method = results_df[metric].idxmax()
    best_value = results_df[metric].max()
    print(f"{metric:15s}: {best_method:20s} ({best_value:.4f})")
print("-" * 60)